In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sopogurjishvili/house-prices-data/sample_submission.csv
/kaggle/input/datasets/sopogurjishvili/house-prices-data/train.csv
/kaggle/input/datasets/sopogurjishvili/house-prices-data/test.csv


In [5]:
!pip install dagshub mlflow xgboost scikit-learn pandas numpy scipy -q

In [6]:
import os
os.environ['MLFLOW_TRACKING_USERNAME'] = 'sophyrise'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '95f86824d28eb66b760d891dfa5f39816ddea9c4'

import dagshub
import mlflow
import pandas as pd
import numpy as np

dagshub.init(repo_owner='sophyrise', repo_name='house-prices-mlflow', mlflow=True)

model = mlflow.sklearn.load_model("models:/HousePricesBestModel/latest")
print("Model loaded successfully!")
print(type(model))

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=eb5b9961-1ab2-4eba-8217-4ba2bdcfbe96&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=83624577c1c2128b8a96defff60993cbea9f5711f1309a725478ecf790e898bc




Output()

Accessing as sophyrise

Initialized MLflow to track repo "sophyrise/house-prices-mlflow"

Repository sophyrise/house-prices-mlflow initialized!

Model loaded successfully!
<class 'xgboost.sklearn.XGBRegressor'>


In [7]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.preprocessing import LabelEncoder

train_orig = pd.read_csv('/kaggle/input/datasets/sopogurjishvili/house-prices-data/train.csv')
test       = pd.read_csv('/kaggle/input/datasets/sopogurjishvili/house-prices-data/test.csv')
test_ids   = test['Id']

# --- Cleaning ---
missing = train_orig.isnull().sum()
missing_pct = (missing / len(train_orig)) * 100
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
cols_to_drop = missing_df[missing_df['pct'] > 80].index.tolist()
train_orig.drop(columns=cols_to_drop, inplace=True)
test.drop(columns=cols_to_drop, inplace=True)

train_orig = train_orig[
    ~((train_orig['GrLivArea'] > 4000) & (train_orig['SalePrice'] < 200000))
].reset_index(drop=True)
train_orig['SalePrice'] = np.log1p(train_orig['SalePrice'])

all_data = pd.concat([train_orig.drop('SalePrice', axis=1), test], axis=0).reset_index(drop=True)

# --- NaN imputation ---
none_cols = ['MasVnrType', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']
for col in none_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2',
             'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(0)

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional']
for col in mode_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))

for col in all_data.columns:
    if all_data[col].isnull().sum() > 0:
        if all_data[col].dtype == 'object':
            all_data[col] = all_data[col].fillna(all_data[col].mode()[0])
        else:
            all_data[col] = all_data[col].fillna(0)

# --- Feature Engineering ---
all_data['TotalSF']     = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalBath']   = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                            all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])
all_data['HouseAge']    = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodelAge']  = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['HasGarage']   = (all_data['GarageArea']  > 0).astype(int)
all_data['HasBsmt']     = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['Has2ndFloor'] = (all_data['2ndFlrSF']    > 0).astype(int)
all_data['QualCond']    = all_data['OverallQual'] * all_data['OverallCond']
all_data['PorchSF']     = (all_data['OpenPorchSF'] + all_data['EnclosedPorch'] +
                            all_data['3SsnPorch']   + all_data['ScreenPorch'])

# --- Ordinal encoding ---
quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond',
             'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']
for col in qual_cols:
    if col in all_data.columns:
        all_data[col] = all_data[col].map(quality_map).fillna(0)

# --- Label encoding ---
le = LabelEncoder()
cat_cols = all_data.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

# --- Skew correction ---
numeric_cols = all_data.select_dtypes(include=[np.number]).columns.tolist()
skewed_feats = all_data[numeric_cols].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75].index.tolist()
all_data[skewed_feats] = np.log1p(all_data[skewed_feats].clip(lower=0))

n_train = len(train_orig)
X_test_final = all_data[n_train:].copy()
if 'Id' in X_test_final.columns:
    X_test_final = X_test_final.drop(columns=['Id'])

print('X_test_final shape:', X_test_final.shape)
print('NaNs:', X_test_final.isnull().sum().sum())

X_test_final shape: (1459, 84)
NaNs: 0


In [8]:
final_features = list(model.feature_names_in_)
X_test_sel = X_test_final[final_features]

final_features = [f for f in final_features if f in X_test_final.columns]
X_test_sel = X_test_final[final_features]

preds_log = model.predict(X_test_sel)
preds     = np.expm1(preds_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print('Submission shape:', submission.shape)

     Id      SalePrice
0  1461  120797.656250
1  1462  151896.406250
2  1463  174639.046875
3  1464  180077.062500
4  1465  194285.781250
5  1466  176569.625000
6  1467  167329.656250
7  1468  166785.750000
8  1469  190216.406250
9  1470  121036.015625
Submission shape: (1459, 2)


In [9]:
print('=== Inference Summary ===')
print(f'Model type      : {type(model).__name__}')
print(f'Loaded from     : HousePricesBestModel (latest)')
print(f'Features used   : {len(final_features)}')
print(f'Predictions     : {len(submission)} houses')
print(f'Price range     : ${submission["SalePrice"].min():,.0f} - ${submission["SalePrice"].max():,.0f}')
print(f'Average price   : ${submission["SalePrice"].mean():,.0f}')
print('Saved to        : submission.csv')

=== Inference Summary ===
Model type      : XGBRegressor
Loaded from     : HousePricesBestModel (latest)
Features used   : 25
Predictions     : 1459 houses
Price range     : $45,596 - $755,810
Average price   : $179,626
Saved to        : submission.csv
